In [1]:
pip uninstall -y torch torchvision torchaudio

Found existing installation: torch 2.6.0+cu124
Uninstalling torch-2.6.0+cu124:
  Successfully uninstalled torch-2.6.0+cu124
Found existing installation: torchvision 0.21.0+cu124
Uninstalling torchvision-0.21.0+cu124:
  Successfully uninstalled torchvision-0.21.0+cu124
Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install torch torchvision --index-url https://download.pytorch.org/whl/cu124

Looking in indexes: https://download.pytorch.org/whl/cu124
  Using cached torch-2.6.0%2Bcu124-cp312-cp312-linux_x86_64.whl.metadata (28 kB)
  Using cached torchvision-0.21.0%2Bcu124-cp312-cp312-linux_x86_64.whl.metadata (6.1 kB)
Using cached torch-2.6.0%2Bcu124-cp312-cp312-linux_x86_64.whl (768.4 MB)
Using cached torchvision-0.21.0%2Bcu124-cp312-cp312-linux_x86_64.whl (7.3 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [torchvision] [torchvision]
Note: you may need to restart the kernel to use updated packages.


In [3]:
# ============================================================
# Install (same stack as bench.ipynb, no torch conflicts)
# Use a FRESH Kaggle GPU session. Internet ON.
# ============================================================
import subprocess, sys

def pipi(*args):
    cmd = [sys.executable, "-m", "pip", "install", "-q", *args]
    print(">>", " ".join(cmd))
    subprocess.check_call(cmd)

pipi("-U", "pip", "wheel")
pipi("shapely", "tqdm", "pillow", "opencv-python-headless", "PyYAML", "lmdb", "pyclipper")

# PaddlePaddle GPU (cu118 wheel — same as bench.ipynb)
try:
    pipi(
        "paddlepaddle-gpu==3.0.0",
        "-i", "https://www.paddlepaddle.org.cn/packages/stable/cu118/",
    )
except subprocess.CalledProcessError:
    print("GPU wheel failed — falling back to CPU paddle (slow but functional)")
    pipi("paddlepaddle")

# langchain stubs needed by paddleocr/paddlex on Kaggle
pipi("langchain-core", "langchain-text-splitters")
pipi("paddleocr")

print("\n Install complete. If 'PDX has already been initialized' appears later, "
      "restart the session and Run All from THIS cell.")

>> /usr/bin/python3 -m pip install -q -U pip wheel
>> /usr/bin/python3 -m pip install -q shapely tqdm pillow opencv-python-headless PyYAML lmdb pyclipper
>> /usr/bin/python3 -m pip install -q paddlepaddle-gpu==3.0.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu118/
>> /usr/bin/python3 -m pip install -q langchain-core langchain-text-splitters
>> /usr/bin/python3 -m pip install -q paddleocr

 Install complete. If 'PDX has already been initialized' appears later, restart the session and Run All from THIS cell.


In [4]:
# ============================================================
# 1. Clone PaddleOCR repo (needed for inference tools)
# ============================================================
import os, subprocess, sys

REPO_DIR = "/kaggle/working/PaddleOCR"
if not os.path.isdir(REPO_DIR):
    subprocess.check_call(["git", "clone", "--depth", "1",
                           "https://github.com/PaddlePaddle/PaddleOCR.git",
                           REPO_DIR])
    print("PaddleOCR repo cloned")
else:
    print("Repo already present")

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print("REPO_DIR:", REPO_DIR)

Repo already present
REPO_DIR: /kaggle/working/PaddleOCR


In [5]:
# ============================================================
# 2. Imports, paths, device
# ============================================================
import os, glob, re, unicodedata, json
from pathlib import Path
from typing import List, Optional, Tuple

os.environ.setdefault("PADDLE_PDX_EAGER_INIT", "0")

import cv2
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import paddle
import torch

from PIL import Image, ImageDraw, ImageFont
from shapely.geometry import Polygon

# ---- VinText dataset paths ----
def _find_vintext_root():
    candidates = [
        "/kaggle/input/datasets/hungkhoi/vietnamese-scene-text-spotting-dataset-vintext/vintext/vintext",
        "/kaggle/input/vietnamese-scene-text-spotting-dataset-vintext/vintext/vintext",
    ]
    for c in candidates:
        if os.path.isdir(os.path.join(c, "test_image")):
            return c
    for base in glob.glob("/kaggle/input/*"):
        tail = os.path.join(base, "vintext", "vintext")
        if os.path.isdir(os.path.join(tail, "test_image")):
            return tail
    return candidates[0]

BASE      = _find_vintext_root()
TEST_DIR  = os.path.join(BASE, "test_image")
VAL_DIR   = os.path.join(BASE, "val_image")
TRAIN_DIR = os.path.join(BASE, "train_images")
LABEL_DIR = os.path.join(BASE, "labels")

OUT_DIR = "/kaggle/working/ppocrv3_inference"
os.makedirs(OUT_DIR, exist_ok=True)

PADDLE_DEVICE = "gpu" if (paddle.device.is_compiled_with_cuda()
                          and paddle.device.cuda.device_count() > 0) else "cpu"
print(f"Paddle device : {PADDLE_DEVICE}")
print(f"Dataset root  : {BASE}")
print(f"Output dir    : {OUT_DIR}")

/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:711: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)


Paddle device : gpu
Dataset root  : /kaggle/input/datasets/hungkhoi/vietnamese-scene-text-spotting-dataset-vintext/vintext/vintext
Output dir    : /kaggle/working/ppocrv3_inference


In [6]:
# ============================================================
# 3. Vietnamese alphabet + annotation helpers
#    Identical to both training notebooks (NFD, 106 chars)
# ============================================================
VN_BASE_CHARS = (
    "abcdefghijklmnopqrstuvwxyz"
    "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
    "0123456789"
    " !\"#$%&'()*+,-./:;<=>?@[\\]^_`{|}~"
    "\u0301"  # acute
    "\u0300"  # grave
    "\u0309"  # hook above
    "\u0323"  # dot below
    "\u0303"  # tilde
    "\u0302"  # circumflex
    "\u0306"  # breve
    "\u031b"  # horn
    "\u0111"  # d with stroke
    "\u0110"  # D with stroke
)
_seen, VN_CHARS = set(), []
for c in VN_BASE_CHARS:
    if c not in _seen:
        VN_CHARS.append(c); _seen.add(c)
print(f"Vietnamese alphabet: {len(VN_CHARS)} chars (NFD)")


def normalize_for_match(s: str) -> str:
    return unicodedata.normalize("NFD", s).strip()


def parse_annotation(gt_path: str) -> List[Tuple[np.ndarray, str]]:
    records = []
    if not os.path.exists(gt_path):
        return records
    with open(gt_path, encoding="utf-8-sig", errors="replace") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split(",", 8)
            if len(parts) < 9:
                continue
            try:
                coords = list(map(float, parts[:8]))
            except ValueError:
                continue
            transcript = parts[8].strip()
            pts = np.array(coords, dtype=np.float32).reshape(4, 2)
            records.append((pts, transcript))
    return records


def _img_idx_from_name(name: str) -> Optional[int]:
    m = re.search(r"(\d+)", os.path.basename(name))
    return int(m.group(1)) if m else None


def order_quad(pts: np.ndarray) -> np.ndarray:
    pts = np.asarray(pts, dtype=np.float32)
    rect = np.zeros((4, 2), dtype=np.float32)
    s = pts.sum(axis=1)
    rect[0] = pts[np.argmin(s)]
    rect[2] = pts[np.argmax(s)]
    diff = pts[:, 1] - pts[:, 0]
    rect[1] = pts[np.argmin(diff)]
    rect[3] = pts[np.argmax(diff)]
    return rect


def crop_quad(image_bgr: np.ndarray, pts: np.ndarray, target_h: int = 48) -> np.ndarray:
    pts = order_quad(pts.astype(np.float32))
    w1 = np.linalg.norm(pts[1] - pts[0]); w2 = np.linalg.norm(pts[2] - pts[3])
    h1 = np.linalg.norm(pts[3] - pts[0]); h2 = np.linalg.norm(pts[2] - pts[1])
    W = max(int((w1 + w2) / 2), 1); H = max(int((h1 + h2) / 2), 1)
    dst = np.array([[0, 0], [W - 1, 0], [W - 1, H - 1], [0, H - 1]], dtype=np.float32)
    M = cv2.getPerspectiveTransform(pts, dst)
    crop = cv2.warpPerspective(image_bgr, M, (W, H))
    scale = target_h / max(crop.shape[0], 1)
    new_w = max(int(crop.shape[1] * scale), 1)
    return cv2.resize(crop, (new_w, target_h))

Vietnamese alphabet: 105 chars (NFD)


In [7]:
# ============================================================
# 4. Locate model files
# ============================================================
MODEL_BASE = "/kaggle/input/models/khnhtraan/pre-trained/pytorch/default/1"

DET_PTH_PATH  = os.path.join(MODEL_BASE, "ppocrv3_mobile_det_vintext.pth")
REC_PTH_PATH  = os.path.join(MODEL_BASE, "ppocrv3_mobile_rec_vintext_12epochs.pth")
BUNDLE_DIR    = os.path.join(MODEL_BASE, "ppocrv3_mobile_rec_vintext_12epochs_paddle_bundle")

# The bundle contains paddle_files/inference/ (exported static graph)
# and paddle_files/vintext_latin_dict.txt
PADDLE_FILES_DIR = os.path.join(BUNDLE_DIR, "paddle_files")
REC_INFER_SRC    = os.path.join(PADDLE_FILES_DIR, "inference")
DICT_SRC         = os.path.join(PADDLE_FILES_DIR, "vintext_latin_dict.txt")

for label, path in [
    ("Det .pth",           DET_PTH_PATH),
    ("Rec .pth",           REC_PTH_PATH),
    ("Bundle dir",         BUNDLE_DIR),
    ("Rec inference dir",  REC_INFER_SRC),
    ("Dict",               DICT_SRC),
]:
    exists = os.path.exists(path)
    status = "OK" if exists else "NOT FOUND"
    print(f"  [{status}] {label}: {path}")

# Copy inference graph + dict to working dir (Paddle can't read from read-only /kaggle/input)
import shutil
REC_INFER_DIR = os.path.join(OUT_DIR, "rec_inference")
DICT_PATH     = os.path.join(OUT_DIR, "vintext_latin_dict.txt")

if os.path.isdir(REC_INFER_SRC) and not os.path.isdir(REC_INFER_DIR):
    shutil.copytree(REC_INFER_SRC, REC_INFER_DIR)
    print(f"Copied rec inference graph -> {REC_INFER_DIR}")
elif os.path.isdir(REC_INFER_DIR):
    print(f"Rec inference dir already present: {REC_INFER_DIR}")
else:
    print("WARNING: rec inference dir not found — E2E pipeline will be unavailable")

if os.path.isfile(DICT_SRC):
    shutil.copy2(DICT_SRC, DICT_PATH)
    with open(DICT_PATH) as f:
        dict_size = sum(1 for _ in f)
    print(f"Dictionary copied: {dict_size} chars -> {DICT_PATH}")
else:
    # Fallback: reconstruct dictionary from VN_CHARS (same order as training)
    print("Dict not found — reconstructing from VN_CHARS")
    with open(DICT_PATH, "w", encoding="utf-8") as f:
        for ch in VN_CHARS:
            f.write(ch + "\n")

  [OK] Det .pth: /kaggle/input/models/khnhtraan/pre-trained/pytorch/default/1/ppocrv3_mobile_det_vintext.pth
  [OK] Rec .pth: /kaggle/input/models/khnhtraan/pre-trained/pytorch/default/1/ppocrv3_mobile_rec_vintext_12epochs.pth
  [OK] Bundle dir: /kaggle/input/models/khnhtraan/pre-trained/pytorch/default/1/ppocrv3_mobile_rec_vintext_12epochs_paddle_bundle
  [OK] Rec inference dir: /kaggle/input/models/khnhtraan/pre-trained/pytorch/default/1/ppocrv3_mobile_rec_vintext_12epochs_paddle_bundle/paddle_files/inference
  [OK] Dict: /kaggle/input/models/khnhtraan/pre-trained/pytorch/default/1/ppocrv3_mobile_rec_vintext_12epochs_paddle_bundle/paddle_files/vintext_latin_dict.txt
Rec inference dir already present: /kaggle/working/ppocrv3_inference/rec_inference
Dictionary copied: 197 chars -> /kaggle/working/ppocrv3_inference/vintext_latin_dict.txt


In [8]:
# ============================================================
# 5. Export PaddleOCR detector from .pth
#    The .pth stores raw Paddle tensor weights (torch container).
#    We convert back to .pdparams, write a minimal YAML config,
#    then call export_model.py to get the inference static graph.
# ============================================================
DET_INFER_DIR  = os.path.join(OUT_DIR, "det_inference")
DET_PDPARAMS   = os.path.join(OUT_DIR, "det_from_pth")
DET_CONFIG     = os.path.join(OUT_DIR, "ppocrv3_det_infer.yml")
INPUT_SIZE     = 640
DUMMY_LABEL    = os.path.join(OUT_DIR, "dummy_label.txt")
open(DUMMY_LABEL, "w").close()

det_yaml = f"""Global:
  use_gpu: {'true' if PADDLE_DEVICE == 'gpu' else 'false'}
  epoch_num: 1
  save_model_dir: {OUT_DIR}
  pretrained_model:
  checkpoints:
  save_inference_dir:
  character_type: ch
  max_text_length: 25
  infer_mode: false
  use_space_char: true
Architecture:
  model_type: det
  algorithm: DB
  Transform:
  Backbone:
    name: MobileNetV3
    scale: 0.5
    model_name: large
    disable_se: true
  Neck:
    name: RSEFPN
    out_channels: 96
    shortcut: true
  Head:
    name: DBHead
    k: 50
Loss:
  name: DBLoss
  balance_loss: true
  main_loss_type: DiceLoss
  alpha: 5
  beta: 10
  ohem_ratio: 3
PostProcess:
  name: DBPostProcess
  thresh: 0.3
  box_thresh: 0.6
  max_candidates: 1000
  unclip_ratio: 1.5
Metric:
  name: DetMetric
  main_indicator: hmean
Train:
  dataset:
    name: SimpleDataSet
    data_dir: /
    label_file_list:
      - {DUMMY_LABEL}
    transforms:
      - DecodeImage:
          img_mode: BGR
          channel_first: false
      - DetLabelEncode:
      - DetResizeForTest:
          image_shape: [{INPUT_SIZE}, {INPUT_SIZE}]
      - NormalizeImage:
          scale: 1./255.
          mean: [0.485, 0.456, 0.406]
          std: [0.229, 0.224, 0.225]
          order: hwc
      - ToCHWImage:
      - KeepKeys:
          keep_keys: [image, shape, polys, ignore_tags]
  loader:
    shuffle: false
    drop_last: false
    batch_size_per_card: 1
    num_workers: 0
Eval:
  dataset:
    name: SimpleDataSet
    data_dir: /
    label_file_list:
      - {DUMMY_LABEL}
    transforms:
      - DecodeImage:
          img_mode: BGR
          channel_first: false
      - DetLabelEncode:
      - DetResizeForTest:
          image_shape: [{INPUT_SIZE}, {INPUT_SIZE}]
      - NormalizeImage:
          scale: 1./255.
          mean: [0.485, 0.456, 0.406]
          std: [0.229, 0.224, 0.225]
          order: hwc
      - ToCHWImage:
      - KeepKeys:
          keep_keys: [image, shape, polys, ignore_tags]
  loader:
    shuffle: false
    drop_last: false
    batch_size_per_card: 1
    num_workers: 0
"""
with open(DET_CONFIG, "w") as f:
    f.write(det_yaml)
print(f"Detector YAML written: {DET_CONFIG}")

# Convert .pth (torch container of Paddle tensors) -> .pdparams
if not os.path.exists(DET_PDPARAMS + ".pdparams"):
    print(f"Converting {DET_PTH_PATH} -> {DET_PDPARAMS}.pdparams ...")
    try:
        ts = torch.load(DET_PTH_PATH, map_location="cpu", weights_only=False)
    except TypeError:
        ts = torch.load(DET_PTH_PATH, map_location="cpu")
    # The det .pth is a plain state_dict of torch tensors
    if isinstance(ts, dict) and "state_dict" in ts:
        ts = ts["state_dict"]
    paddle_state = {
        k: paddle.to_tensor(v.detach().cpu().numpy() if hasattr(v, "detach") else np.asarray(v))
        for k, v in ts.items()
    }
    paddle.save(paddle_state, DET_PDPARAMS + ".pdparams")
    print(f"  Saved {len(paddle_state)} tensors -> {DET_PDPARAMS}.pdparams")
else:
    print(f"det .pdparams already exists: {DET_PDPARAMS}.pdparams")

# Export static inference graph
os.makedirs(DET_INFER_DIR, exist_ok=True)
export_cmd = [
    sys.executable, os.path.join(REPO_DIR, "tools", "export_model.py"),
    "-c", DET_CONFIG,
    "-o",
    f"Global.pretrained_model={DET_PDPARAMS}",
    f"Global.save_inference_dir={DET_INFER_DIR}",
    f"Global.use_gpu={'true' if PADDLE_DEVICE == 'gpu' else 'false'}",
]
print("Exporting detector inference graph...")
result = subprocess.run(export_cmd, cwd=REPO_DIR, capture_output=True, text=True)
if result.returncode != 0:
    print("EXPORT STDERR:", result.stderr[-2000:])
    raise RuntimeError("Detector export failed")
print(f"Detector exported -> {DET_INFER_DIR}")
print(os.listdir(DET_INFER_DIR))

Detector YAML written: /kaggle/working/ppocrv3_inference/ppocrv3_det_infer.yml
det .pdparams already exists: /kaggle/working/ppocrv3_inference/det_from_pth.pdparams
Exporting detector inference graph...
Detector exported -> /kaggle/working/ppocrv3_inference/det_inference
['inference.pdiparams', 'inference.json', 'inference.yml']


In [22]:
# ============================================================
# 6. Build PaddleOCR detector and recogniser inference callables
# ============================================================
def _allow_legacy_yml(model_dir: str) -> None:
    """Strip model_name from inference.yml if present (breaks older Paddle versions)."""
    yml_path = os.path.join(model_dir, "inference.yml")
    if not os.path.isfile(yml_path):
        return
    import yaml
    with open(yml_path, encoding="utf-8") as f:
        cfg = yaml.safe_load(f) or {}
    g = cfg.setdefault("Global", {})
    if g.get("model_name"):
        g["model_name"] = ""
        with open(yml_path, "w", encoding="utf-8") as f:
            yaml.safe_dump(cfg, f, allow_unicode=True, default_flow_style=False)


def _make_det_predictor(model_dir: str, use_gpu: bool = True):
    from tools.infer import utility
    from tools.infer.predict_det import TextDetector
    old_argv = sys.argv
    sys.argv = [
        "predict_det.py",
        "--det_model_dir", model_dir,
        "--use_gpu", "true" if use_gpu else "false",
        "--det_algorithm", "DB",
        "--det_box_type", "quad",
    ]
    args = utility.parse_args()
    sys.argv = old_argv
    detector = TextDetector(args)

    def _pred(img_bgr: np.ndarray) -> List[np.ndarray]:
        dt_boxes, _ = detector(img_bgr)
        if dt_boxes is None or len(dt_boxes) == 0:
            return []
        return [np.asarray(b, dtype=np.float32) for b in dt_boxes]

    return _pred, detector


def _make_rec_predictor(model_dir: str, dict_path: str, use_gpu: bool = True):
    _allow_legacy_yml(model_dir)
    from tools.infer import utility
    from tools.infer.predict_rec import TextRecognizer
    old_argv = sys.argv
    sys.argv = [
        "predict_rec.py",
        "--rec_model_dir", model_dir,
        "--rec_char_dict_path", dict_path,
        "--use_gpu", "true" if use_gpu else "false",
        "--use_space_char", "true",
        "--rec_algorithm", "SVTR_LCNet",
        "--rec_image_shape", "3,48,320",
    ]
    args = utility.parse_args()
    sys.argv = old_argv
    recognizer = TextRecognizer(args)

    def _pred_crop(crop_bgr: np.ndarray) -> str:
        rec_res, _ = recognizer([crop_bgr])
        if not rec_res:
            return ""
        item = rec_res[0]
        if isinstance(item, (list, tuple)):
            return item[0] if item else ""
        return str(item)

    return _pred_crop, recognizer


USE_GPU = (PADDLE_DEVICE == "gpu")

print("Loading detector...")
det_predict, DETECTOR = _make_det_predictor(DET_INFER_DIR, use_gpu=USE_GPU)
print("  Detector ready.")

rec_predict = None
RECOGNIZER  = None
if os.path.isdir(REC_INFER_DIR):
    print("Loading recogniser...")
    rec_predict, RECOGNIZER = _make_rec_predictor(REC_INFER_DIR, DICT_PATH, use_gpu=USE_GPU)
    print("  Recogniser ready.")
else:
    print("[SKIP] Recogniser inference dir not found — E2E pipeline unavailable.")

Loading detector...
[2026/05/31 10:49:13] ppocr WARNING: The first GPU is used for inference by default, GPU ID: 0
  Detector ready.
Loading recogniser...
[2026/05/31 10:49:13] ppocr WARNING: The first GPU is used for inference by default, GPU ID: 0
  Recogniser ready.


In [23]:
# ============================================================
# 7. E2E pipeline wrapper
# ============================================================
class PaddleE2EPipeline:
    """Wraps det_predict + rec_predict into a single call.
    Returns List[(quad_np, text_str)] for one image.
    """
    def __init__(self, det_fn, rec_fn, min_crop_w: int = 4):
        self.det_fn = det_fn
        self.rec_fn = rec_fn
        self.min_crop_w = min_crop_w

    def __call__(self, image_bgr: np.ndarray) -> List[Tuple[np.ndarray, str]]:
        spotted = []
        for poly in self.det_fn(image_bgr):
            if poly is None or len(poly) < 4:
                continue
            try:
                crop = crop_quad(image_bgr, poly.astype(np.float32), target_h=48)
            except Exception:
                continue
            if crop.shape[1] < self.min_crop_w:
                continue
            text = self.rec_fn(crop) if self.rec_fn else ""
            spotted.append((poly.astype(np.float32), text))
        return spotted


det_only_pipeline = None  # returns [(quad, score=None)]
e2e_pipeline      = None

if det_predict is not None:
    # Detection-only wrapper: returns List[(quad,)]
    def _det_only(img_bgr):
        return [(p,) for p in det_predict(img_bgr)]
    det_only_pipeline = _det_only

if det_predict is not None and rec_predict is not None:
    e2e_pipeline = PaddleE2EPipeline(det_predict, rec_predict)

print(f"det_only_pipeline : {'ready' if det_only_pipeline else 'unavailable'}")
print(f"e2e_pipeline      : {'ready' if e2e_pipeline else 'unavailable'}")

det_only_pipeline : ready
e2e_pipeline      : ready


In [11]:
# ============================================================
# 8. Download NotoSans font for Vietnamese text rendering
# ============================================================
FONT_PATH = os.path.join(OUT_DIR, "NotoSans-Regular.ttf")
FONT_URL  = "https://github.com/googlefonts/noto-fonts/raw/main/hinted/ttf/NotoSans/NotoSans-Regular.ttf"

if not os.path.exists(FONT_PATH):
    try:
        subprocess.run(["wget", "-q", "-O", FONT_PATH, FONT_URL], check=True)
        print(f"Font downloaded -> {FONT_PATH}")
    except Exception as e:
        print(f"Font download failed ({e}); will fall back to PIL default.")
else:
    print(f"Font present: {FONT_PATH}")


def _font(size: int) -> ImageFont.FreeTypeFont:
    try:
        return ImageFont.truetype(FONT_PATH, size)
    except Exception:
        return ImageFont.load_default()


def draw_text_pil(canvas_bgr: np.ndarray,
                  text: str,
                  position: Tuple[int, int],
                  color_rgb: Tuple[int, int, int],
                  font_size: int = 18,
                  bg_rgb: Tuple[int, int, int] = (0, 0, 0)) -> np.ndarray:
    """Render text onto a BGR numpy image using PIL/NotoSans with background box."""
    pil  = Image.fromarray(cv2.cvtColor(canvas_bgr, cv2.COLOR_BGR2RGB))
    draw = ImageDraw.Draw(pil)
    font = _font(font_size)
    bbox = draw.textbbox(position, text, font=font)
    pad  = 2
    draw.rectangle([bbox[0]-pad, bbox[1]-pad, bbox[2]+pad, bbox[3]+pad], fill=bg_rgb)
    draw.text(position, text, font=font, fill=color_rgb)
    return cv2.cvtColor(np.array(pil), cv2.COLOR_RGB2BGR)

Font present: /kaggle/working/ppocrv3_inference/NotoSans-Regular.ttf


In [13]:
# ============================================================
# 9. Visualisation helpers
#    Left  : original + GT boxes (green) + GT text
#    Right : predicted boxes (cyan) + recognised text
# ============================================================
GT_BOX_BGR    = (0,   200,   0)   # green
GT_TEXT_RGB   = (0,   230,   0)
GT_TEXTBG     = (0,    40,   0)

PRED_BOX_BGR  = (255, 200,   0)   # cyan
PRED_TEXT_RGB = (0,   215, 255)
PRED_TEXTBG   = (20,   20,  80)


def _annotate_gt(img_bgr: np.ndarray,
                 gt: List[Tuple[np.ndarray, str]],
                 font_size: int) -> np.ndarray:
    canvas = img_bgr.copy()
    for poly, txt in gt:
        if txt in ("###", "***", ""):
            continue
        cv2.polylines(canvas, [poly.astype(np.int32)], True, GT_BOX_BGR, 2)
        x = int(poly[:, 0].min())
        y = int(poly[:, 1].min())
        label_y = max(font_size + 4, y - font_size - 4)
        canvas = draw_text_pil(
            canvas, f"GT: {txt}",
            (x, label_y),
            color_rgb=GT_TEXT_RGB,
            font_size=font_size,
            bg_rgb=GT_TEXTBG,
        )
    return canvas


def _annotate_det_pred(img_bgr: np.ndarray,
                       polys: List[np.ndarray],
                       font_size: int) -> np.ndarray:
    """Detection-only: draw boxes with index label."""
    canvas = img_bgr.copy()
    for i, poly in enumerate(polys):
        cv2.polylines(canvas, [poly.astype(np.int32)], True, PRED_BOX_BGR, 2)
        x = int(poly[:, 0].min())
        y = int(poly[:, 1].min())
        label_y = max(font_size + 4, y - font_size - 2)
        canvas = draw_text_pil(
            canvas, f"#{i+1}",
            (x, label_y),
            color_rgb=PRED_TEXT_RGB,
            font_size=font_size,
            bg_rgb=PRED_TEXTBG,
        )
    return canvas


def _annotate_e2e_pred(img_bgr: np.ndarray,
                       preds: List[Tuple[np.ndarray, str]],
                       font_size: int) -> np.ndarray:
    """E2E: draw boxes with recognised text."""
    canvas = img_bgr.copy()
    for poly, text in preds:
        cv2.polylines(canvas, [poly.astype(np.int32)], True, PRED_BOX_BGR, 2)
        x = int(poly[:, 0].min())
        y = int(poly[:, 1].min())
        label_y = max(font_size + 4, y - font_size - 2)
        canvas = draw_text_pil(
            canvas, f"PR: {text or '?'}",
            (x, label_y),
            color_rgb=PRED_TEXT_RGB,
            font_size=font_size,
            bg_rgb=PRED_TEXTBG,
        )
    return canvas


def save_side_by_side(gt_canvas: np.ndarray,
                      pred_canvas: np.ndarray,
                      title_left: str,
                      title_right: str,
                      save_path: str,
                      suptitle: str = "") -> None:
    fig, axes = plt.subplots(1, 2, figsize=(22, 11))
    axes[0].imshow(cv2.cvtColor(gt_canvas,   cv2.COLOR_BGR2RGB))
    axes[0].set_title(title_left,  fontsize=13, fontweight="bold")
    axes[0].axis("off")
    axes[1].imshow(cv2.cvtColor(pred_canvas, cv2.COLOR_BGR2RGB))
    axes[1].set_title(title_right, fontsize=13, fontweight="bold")
    axes[1].axis("off")
    if suptitle:
        plt.suptitle(suptitle, fontsize=12)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  saved -> {save_path}")

In [14]:
# ============================================================
# 10. Locate the specific target images
#     im1513, im1558, im1626, im1828, im1880
#     Search test_image first, then val_image, then train_images.
# ============================================================
TARGET_STEMS = ["im1513", "im1558", "im1626", "im1828", "im1880"]

def _find_images(stems: List[str]) -> dict:
    """Return {stem: full_path} by searching test -> val -> train dirs."""
    found = {}
    for split_dir in [TEST_DIR, VAL_DIR, TRAIN_DIR]:
        for ext in (".jpg", ".png"):
            for stem in stems:
                if stem not in found:
                    candidate = os.path.join(split_dir, stem + ext)
                    if os.path.isfile(candidate):
                        found[stem] = candidate
    return found

target_images = _find_images(TARGET_STEMS)
print("Target images located:")
for stem in TARGET_STEMS:
    path = target_images.get(stem)
    print(f"  {stem}: {path or 'NOT FOUND'}")

Target images located:
  im1513: /kaggle/input/datasets/hungkhoi/vietnamese-scene-text-spotting-dataset-vintext/vintext/vintext/test_image/im1513.jpg
  im1558: /kaggle/input/datasets/hungkhoi/vietnamese-scene-text-spotting-dataset-vintext/vintext/vintext/test_image/im1558.jpg
  im1626: /kaggle/input/datasets/hungkhoi/vietnamese-scene-text-spotting-dataset-vintext/vintext/vintext/test_image/im1626.jpg
  im1828: /kaggle/input/datasets/hungkhoi/vietnamese-scene-text-spotting-dataset-vintext/vintext/vintext/test_image/im1828.jpg
  im1880: /kaggle/input/datasets/hungkhoi/vietnamese-scene-text-spotting-dataset-vintext/vintext/vintext/test_image/im1880.jpg


In [17]:
# ============================================================
# 11a. PPOCRv3 Detection-only inference
#      Left : original + GT boxes (green) + GT labels
#      Right: predicted boxes (cyan) + box index
# ============================================================
if det_only_pipeline is None:
    print("[SKIP] Detector not ready.")
else:
    viz_det_dir = os.path.join(OUT_DIR, "det_only_samples")
    os.makedirs(viz_det_dir, exist_ok=True)
    print("=== PPOCRv3 Detection-Only Inference ===")

    for stem in TARGET_STEMS:
        ip = target_images.get(stem)
        if ip is None:
            print(f"  [SKIP] {stem}: image not found")
            continue

        img = cv2.imread(ip)
        if img is None:
            print(f"  [SKIP] {stem}: cv2.imread failed")
            continue

        idx = _img_idx_from_name(ip)
        gt  = parse_annotation(os.path.join(LABEL_DIR, f"gt_{idx}.txt"))

        font_size = max(14, img.shape[0] // 55)

        # Left: GT
        gt_canvas = _annotate_gt(img, gt, font_size)

        # Right: det-only prediction
        pred_polys = det_predict(img)
        pred_canvas = _annotate_det_pred(img, pred_polys, font_size)

        gt_count   = sum(1 for _, t in gt if t not in ("###", "***", ""))
        save_path  = os.path.join(viz_det_dir, f"det_{stem}.png")
        save_side_by_side(
            gt_canvas, pred_canvas,
            title_left  = f"GT ({gt_count} boxes, green)  —  {stem}",
            title_right = f"PPOCRv3 Det predicted ({len(pred_polys)} boxes, cyan)",
            save_path   = save_path,
            suptitle    = stem,
        )

    print(f"\nDetection-only visualisations -> {viz_det_dir}")

=== PPOCRv3 Detection-Only Inference ===
  saved -> /kaggle/working/ppocrv3_inference/det_only_samples/det_im1513.png
  saved -> /kaggle/working/ppocrv3_inference/det_only_samples/det_im1558.png
  saved -> /kaggle/working/ppocrv3_inference/det_only_samples/det_im1626.png
  saved -> /kaggle/working/ppocrv3_inference/det_only_samples/det_im1828.png
  saved -> /kaggle/working/ppocrv3_inference/det_only_samples/det_im1880.png

Detection-only visualisations -> /kaggle/working/ppocrv3_inference/det_only_samples


In [24]:
# ============================================================
# 11b. PPOCRv3 E2E (Det + Rec) inference
#      Left : original + GT boxes (green) + GT text
#      Right: predicted boxes (cyan) + recognised Vietnamese text
# ============================================================
if e2e_pipeline is None:
    print("[SKIP] E2E pipeline not ready (recogniser missing).")
else:
    viz_e2e_dir = os.path.join(OUT_DIR, "e2e_samples")
    os.makedirs(viz_e2e_dir, exist_ok=True)
    print("=== PPOCRv3 E2E (Det + Rec) Inference ===")

    for stem in TARGET_STEMS:
        ip = target_images.get(stem)
        if ip is None:
            print(f"  [SKIP] {stem}: image not found")
            continue

        img = cv2.imread(ip)
        if img is None:
            print(f"  [SKIP] {stem}: cv2.imread failed")
            continue

        idx = _img_idx_from_name(ip)
        gt  = parse_annotation(os.path.join(LABEL_DIR, f"gt_{idx}.txt"))

        font_size = max(14, img.shape[0] // 55)

        # Left: GT
        gt_canvas = _annotate_gt(img, gt, font_size)

        # Right: E2E predictions
        preds = e2e_pipeline(img)   # List[(quad, text)]
        pred_canvas = _annotate_e2e_pred(img, preds, font_size)

        gt_count  = sum(1 for _, t in gt if t not in ("###", "***", ""))
        save_path = os.path.join(viz_e2e_dir, f"e2e_{stem}.png")
        save_side_by_side(
            gt_canvas, pred_canvas,
            title_left  = f"GT ({gt_count} boxes, green)  —  {stem}",
            title_right = f"PPOCRv3 E2E predicted ({len(preds)} boxes, cyan)",
            save_path   = save_path,
            suptitle    = stem,
        )

        # Also print recognised texts to console
        print(f"  {stem}: {len(preds)} detections")
        for poly, text in preds:
            print(f"    -> '{text}'")

    print(f"\nE2E visualisations -> {viz_e2e_dir}")

=== PPOCRv3 E2E (Det + Rec) Inference ===
  saved -> /kaggle/working/ppocrv3_inference/e2e_samples/e2e_im1513.png
  im1513: 9 detections
    -> '100'
    -> '%'
    -> 'NGUYÊN'
    -> 'CHẤT'
    -> 'ITRUNG'
    -> 'XAY'
    -> 'RANG'
    -> 'PHÊ'
    -> 'CÀ'
  saved -> /kaggle/working/ppocrv3_inference/e2e_samples/e2e_im1558.png
  im1558: 19 detections
    -> 'ĐẠI'
    -> 'CUỘC'
    -> 'HỘI'
    -> 'VÀO'
    -> 'ĐI'
    -> 'XI'
    -> 'SỐNG'
    -> 'ĐẢNG'
    -> 'CỦA'
    -> 'NGHỊ'
    -> 'ĐƯA'
    -> 'TÂM'
    -> 'QUYẾT'
    -> 'QUYẾT'
    -> 'THÀNH'
    -> 'MINH'
    -> 'PHỐ'
    -> 'CHÍ'
    -> 'HỒ'
  saved -> /kaggle/working/ppocrv3_inference/e2e_samples/e2e_im1626.png
  im1626: 16 detections
    -> 'GREEN'
    -> 'TFA'
    -> 'Cog'
    -> 'Cao'
    -> 'cấp'
    -> 'Thái'
    -> 'Nguyên'
    -> 'Trà'
    -> 'Xanh'
    -> 'bạn!'
    -> 'cấp'
    -> 'của'
    -> 'Đẳng'
    -> 'Gia'
    -> 'Đại'
    -> 'Trả'
  saved -> /kaggle/working/ppocrv3_i

In [ ]:
# ============================================================
# 12. Inline preview of saved results
# ============================================================
import matplotlib.image as mpimg

for label, subdir in [("PPOCRv3 Detection-only", "det_only_samples"),
                      ("PPOCRv3 E2E",            "e2e_samples")]:
    d = os.path.join(OUT_DIR, subdir)
    pngs = sorted(glob.glob(os.path.join(d, "*.png")))
    if not pngs:
        continue
    print(f"\n--- {label} ({len(pngs)} images) ---")
    for png in pngs:
        fig, ax = plt.subplots(1, 1, figsize=(20, 9))
        ax.imshow(mpimg.imread(png))
        ax.set_title(f"{label}  —  {os.path.basename(png)}", fontsize=13)
        ax.axis("off")
        plt.tight_layout()
        plt.show()
        print(f"  {png}")

In [ ]:
# ============================================================
# 13. Summary
# ============================================================
print("\n" + "=" * 70)
print("  PPOCRv3 Inference complete.")
print(f"  Target images : {TARGET_STEMS}")
print(f"  Output root   : {OUT_DIR}")
for subdir in ["det_only_samples", "e2e_samples"]:
    d = os.path.join(OUT_DIR, subdir)
    pngs = glob.glob(os.path.join(d, "*.png"))
    if pngs:
        print(f"    {subdir}/  ({len(pngs)} images)")
print("=" * 70)

# Free GPU memory
if PADDLE_DEVICE == "gpu":
    try:
        paddle.device.cuda.empty_cache()
    except Exception:
        pass